# Module 01: Foundations of Modern GenAI

**Companion notebook for**: `01-foundations-of-genai.html`

## Learning Objectives
- Understand the Transformer architecture and why it replaced RNNs
- Implement scaled dot-product attention and multi-head attention from scratch in NumPy
- Explore tokenization with tiktoken and understand BPE, token costs, and multilingual efficiency
- Generate and visualize OpenAI embeddings, compute cosine similarity, and perform vector arithmetic
- Understand the three-stage training pipeline: pre-training, instruction tuning (SFT), and RLHF
- Implement and compare inference sampling strategies: temperature, top-k, top-p, and greedy decoding
- Estimate KV cache memory requirements for different model architectures

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q tiktoken openai numpy matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Verify API key is available
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
print("All imports successful. OpenAI API key found.")

---
## Section 1: The Transformer Architecture

The **Transformer** (Vaswani et al., 2017 — "Attention Is All You Need") replaced RNNs by allowing every token to attend to every other token simultaneously. This gives two key advantages:

1. **Long-range dependencies** — Token 1 and token 100 can interact in a single step (no degrading memory).
2. **Parallelism** — All positions are computed simultaneously on GPU, making training dramatically faster.

Modern LLMs (GPT-4, Claude, Llama) are **decoder-only** transformers trained to predict the next token. Each transformer layer consists of:
- **Multi-Head Self-Attention** — each token computes relevance scores against all other tokens
- **Feed-Forward Network (FFN)** — position-wise expansion and contraction
- **Residual connections + LayerNorm** — enable training very deep networks (6 to 96+ layers)

### 1.1 Scaled Dot-Product Attention (NumPy)

The core attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$$

- **Q (Query)** — "What am I looking for?"
- **K (Key)** — "What do I contain?"
- **V (Value)** — "What information should I pass on?"

The dot product measures similarity in learned vector space. Division by $\sqrt{d_k}$ prevents softmax from producing extremely peaked distributions in high dimensions.

In [ ]:
import numpy as np


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.

    Q: (seq_len, d_k)  — query vectors
    K: (seq_len, d_k)  — key vectors
    V: (seq_len, d_v)  — value vectors
    mask: optional (seq_len, seq_len) boolean — True = mask out (future tokens)
    Returns: (context, weights)
        context: (seq_len, d_v) — weighted sum of values
        weights: (seq_len, seq_len) — attention weights
    """
    d_k = Q.shape[-1]

    # Step 1: Dot product of Q with K^T -> raw attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)

    # Step 2: Apply causal mask (decoder only) — future tokens get -inf
    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    # Step 3: Softmax over last axis -> attention weights (sum to 1)
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    # Step 4: Weighted sum of values
    output = weights @ V  # (seq_len, d_v)
    return output, weights


# --- Example: simulate a single attention head ---
np.random.seed(42)
seq_len, d_model, d_k = 6, 512, 64

# Token embeddings (randomly initialized for demo)
x = np.random.randn(seq_len, d_model)

# Learned weight matrices (small init like real models)
Wq = np.random.randn(d_model, d_k) * 0.02
Wk = np.random.randn(d_model, d_k) * 0.02
Wv = np.random.randn(d_model, d_k) * 0.02

Q = x @ Wq  # (6, 64)
K = x @ Wk  # (6, 64)
V = x @ Wv  # (6, 64)

# Causal mask: True where future positions should be hidden
mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

context, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
print(f"Context shape: {context.shape}")  # (6, 64)
print(f"Attention weights (row 3): {attn_weights[3].round(3)}")
print("Token 3 can only attend to tokens 0,1,2,3 — future tokens are zeroed out.")

### 1.2 Visualizing the Attention Matrix

The causal mask creates a lower-triangular attention pattern: each token can only attend to itself and previous tokens. This is how decoder-only models (GPT, Claude, Llama) enforce autoregressive generation.

In [ ]:
token_labels = ["The", "cat", "sat", "on", "the", "mat"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: causal (masked) attention
im1 = axes[0].imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Causal (Masked) Attention Weights", fontsize=13)
axes[0].set_xticks(range(seq_len))
axes[0].set_xticklabels(token_labels, fontsize=10)
axes[0].set_yticks(range(seq_len))
axes[0].set_yticklabels(token_labels, fontsize=10)
axes[0].set_xlabel("Key (attends to)")
axes[0].set_ylabel("Query (from)")
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Annotate values
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[i, j]
        if val > 0.01:
            axes[0].text(j, i, f"{val:.2f}", ha="center", va="center",
                         fontsize=8, color="white" if val > 0.5 else "black")

# Right: unmasked attention (bidirectional, like BERT encoder)
context_nomask, weights_nomask = scaled_dot_product_attention(Q, K, V, mask=None)
im2 = axes[1].imshow(weights_nomask, cmap="Oranges", vmin=0, vmax=1)
axes[1].set_title("Bidirectional (Unmasked) Attention Weights", fontsize=13)
axes[1].set_xticks(range(seq_len))
axes[1].set_xticklabels(token_labels, fontsize=10)
axes[1].set_yticks(range(seq_len))
axes[1].set_yticklabels(token_labels, fontsize=10)
axes[1].set_xlabel("Key (attends to)")
axes[1].set_ylabel("Query (from)")
plt.colorbar(im2, ax=axes[1], shrink=0.8)

for i in range(seq_len):
    for j in range(seq_len):
        val = weights_nomask[i, j]
        if val > 0.01:
            axes[1].text(j, i, f"{val:.2f}", ha="center", va="center",
                         fontsize=8, color="white" if val > 0.5 else "black")

plt.tight_layout()
plt.show()

print("Left: Decoder-only (GPT/Claude/Llama) — each token sees only past tokens.")
print("Right: Encoder (BERT) — each token sees all tokens (bidirectional).")

### 1.3 Multi-Head Attention

Multi-head attention runs `h` attention heads in parallel, each with its own learned Q/K/V projections to a lower-dimensional subspace (`d_k = d_model / h`). The outputs are concatenated and projected back to `d_model`.

Each head can specialize in a different type of relationship: subject-verb agreement, coreference, positional proximity, syntactic structure, etc. These specializations emerge from training — they are not designed.

In [ ]:
def multi_head_attention(x, n_heads, d_model):
    """
    Multi-head attention from scratch.

    x: (seq_len, d_model) — input token embeddings
    n_heads: number of attention heads
    d_model: model dimension
    Returns: (seq_len, d_model) — output after multi-head attention
    """
    d_k = d_model // n_heads
    seq_len = x.shape[0]

    # Create random weight matrices for each head
    # In a real model, these are learned parameters
    heads_output = []
    all_weights = []

    for h in range(n_heads):
        np.random.seed(h + 100)  # Reproducible per head
        Wq = np.random.randn(d_model, d_k) * 0.02
        Wk = np.random.randn(d_model, d_k) * 0.02
        Wv = np.random.randn(d_model, d_k) * 0.02

        Q = x @ Wq
        K = x @ Wk
        V = x @ Wv

        # Causal mask for decoder
        mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
        head_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        heads_output.append(head_out)
        all_weights.append(weights)

    # Concatenate all heads: (seq_len, n_heads * d_k) = (seq_len, d_model)
    concat = np.concatenate(heads_output, axis=-1)

    # Output projection W_O: (d_model, d_model)
    np.random.seed(999)
    W_O = np.random.randn(d_model, d_model) * 0.02
    output = concat @ W_O

    return output, all_weights


# Run multi-head attention with 8 heads
np.random.seed(42)
seq_len, d_model = 6, 64  # Small for demo
x = np.random.randn(seq_len, d_model)

mha_output, head_weights = multi_head_attention(x, n_heads=8, d_model=d_model)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {mha_output.shape}")
print(f"Number of heads: {len(head_weights)}")
print(f"Each head's attention matrix shape: {head_weights[0].shape}")

In [ ]:
# Visualize attention patterns across different heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
token_labels = ["The", "cat", "sat", "on", "the", "mat"]

for idx, ax in enumerate(axes.flat):
    im = ax.imshow(head_weights[idx], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"Head {idx}", fontsize=11)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(token_labels, fontsize=8, rotation=45)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(token_labels, fontsize=8)

plt.suptitle("Attention Patterns Across 8 Heads\n(Each head learns different relationships)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Notice how different heads develop different attention patterns.")
print("In trained models, heads specialize in: syntax, coreference, positional proximity, etc.")

### 1.4 Full Transformer Block

A complete transformer block wraps attention and FFN in residual connections with LayerNorm:

```
x -> LayerNorm -> Multi-Head Attention -> + (residual) -> LayerNorm -> FFN -> + (residual) -> output
```

- **Residual connections** (`x + sublayer(x)`) provide a gradient highway for training deep networks.
- **LayerNorm** stabilizes activations to zero mean and unit variance.
- **FFN** expands dimension by 4x then contracts back, acting as a key-value memory for facts.

In [ ]:
def layer_norm(x, eps=1e-5):
    """Layer normalization: zero mean, unit variance per token."""
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


def gelu(x):
    """GELU activation (used in GPT-2, BERT, and most modern transformers)."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


def feed_forward(x, d_model, d_ff):
    """Position-wise feed-forward network: expand by 4x, then contract."""
    np.random.seed(200)
    W1 = np.random.randn(d_model, d_ff) * 0.02
    W2 = np.random.randn(d_ff, d_model) * 0.02
    return gelu(x @ W1) @ W2


def transformer_block(x, n_heads, d_model):
    """
    One full transformer decoder block:
    LayerNorm -> MHA -> Residual -> LayerNorm -> FFN -> Residual
    """
    d_ff = d_model * 4  # Standard 4x expansion

    # Sub-layer 1: Multi-Head Attention with residual
    normed = layer_norm(x)
    attn_out, _ = multi_head_attention(normed, n_heads, d_model)
    x = x + attn_out  # Residual connection

    # Sub-layer 2: FFN with residual
    normed = layer_norm(x)
    ffn_out = feed_forward(normed, d_model, d_ff)
    x = x + ffn_out  # Residual connection

    return x


# Run a full transformer block
np.random.seed(42)
seq_len, d_model = 6, 64
x = np.random.randn(seq_len, d_model)

output = transformer_block(x, n_heads=8, d_model=d_model)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input norm:   {np.linalg.norm(x):.3f}")
print(f"Output norm:  {np.linalg.norm(output):.3f}")
print("\nThe residual connections keep the output magnitude similar to the input.")
print("Without residuals, gradients would vanish through 96 layers.")

### 1.5 Comparing Model Architectures

| Architecture | Examples | Attention | Use Case |
|---|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Bidirectional (sees all tokens) | Classification, NER, embeddings |
| **Decoder-only** | GPT-4, Claude, Llama | Causal (sees only past tokens) | Text generation, chat, code |
| **Encoder-Decoder** | T5, BART, original Transformer | Both | Translation, summarization |

In [ ]:
# Compare model sizes and architectures
models = [
    {"name": "BERT-base",     "arch": "Encoder",     "layers": 12,  "d_model": 768,   "heads": 12, "params": "110M"},
    {"name": "GPT-2",         "arch": "Decoder",     "layers": 12,  "d_model": 768,   "heads": 12, "params": "117M"},
    {"name": "T5-base",       "arch": "Enc-Dec",     "layers": 12,  "d_model": 768,   "heads": 12, "params": "220M"},
    {"name": "GPT-3",         "arch": "Decoder",     "layers": 96,  "d_model": 12288, "heads": 96, "params": "175B"},
    {"name": "Llama 3 8B",    "arch": "Decoder",     "layers": 32,  "d_model": 4096,  "heads": 32, "params": "8B"},
    {"name": "Llama 3 70B",   "arch": "Decoder",     "layers": 80,  "d_model": 8192,  "heads": 64, "params": "70B"},
    {"name": "GPT-4 (est.)",  "arch": "Decoder",     "layers": 96,  "d_model": 12288, "heads": 96, "params": "~1.8T"},
]

print(f"{'Model':<18} {'Architecture':<12} {'Layers':>6} {'d_model':>8} {'Heads':>6} {'d_k':>6} {'Params':>8}")
print("-" * 72)
for m in models:
    d_k = m["d_model"] // m["heads"]
    print(f"{m['name']:<18} {m['arch']:<12} {m['layers']:>6} {m['d_model']:>8} {m['heads']:>6} {d_k:>6} {m['params']:>8}")

print("\nKey insight: The architecture is essentially the same from BERT to GPT-4.")
print("What changed is scale — more layers, larger dimensions, more data.")

---
## Section 2: Tokenization

Neural networks operate on numbers, not text. **Tokenization** converts text into a sequence of integer IDs that the model can process.

Modern tokenizers use **subword tokenization** (BPE, WordPiece, SentencePiece) — pieces larger than characters but smaller than words. This handles rare words, code, multiple languages, and special characters gracefully.

Key facts:
- Tokens are NOT words. "tokenization" might be `["token", "ization"]`.
- 1 token is approximately 4 characters or 0.75 words in English.
- You pay per token — always count tokens before sending API requests.
- English is 2-5x more token-efficient than most other languages.

### 2.1 Tokenization with tiktoken

In [ ]:
import tiktoken

# Load OpenAI's tokenizer for GPT-4 / GPT-4o
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 encoding

# --- Basic tokenization ---
text = "Hello, world! Tokenization is surprisingly nuanced."
tokens = enc.encode(text)
print(f"Text:        {text}")
print(f"Token IDs:   {tokens}")
print(f"Token count: {len(tokens)}")
print()

# --- Decode individual tokens to see what each one represents ---
print("Individual tokens:")
for tid in tokens:
    piece = enc.decode([tid])
    print(f"  {tid:6d}  ->  {repr(piece)}")

print(f"\nRoundtrip check: {enc.decode(tokens) == text}")

### 2.2 Token Efficiency Across Languages

BPE tokenizers are trained primarily on English text. This means other languages require significantly more tokens to express the same content — increasing cost, reducing context window capacity, and compounding training data disadvantages.

In [ ]:
# Compare token counts across languages and content types
samples = {
    "English":  "The transformer architecture revolutionized natural language processing.",
    "Spanish":  "La arquitectura del transformador revoluciono el procesamiento del lenguaje.",
    "Chinese":  "Transformer架构彻底改变了自然语言处理领域。",
    "Arabic":   "أحدث هيكل المحوّل ثورة في معالجة اللغة الطبيعية.",
    "Code":     "def scaled_attention(Q, K, V):\n    return softmax(Q @ K.T / sqrt(d_k)) @ V",
}

print(f"{'Language':<10} {'Tokens':>6} {'Chars':>6} {'Chars/Tok':>10}")
print("-" * 38)

token_counts = []
languages = []
for lang, sample in samples.items():
    n = len(enc.encode(sample))
    chars = len(sample)
    token_counts.append(n)
    languages.append(lang)
    print(f"{lang:<10} {n:>6} {chars:>6} {chars/n:>10.1f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#3b82f6", "#06b6d4", "#f97316", "#a855f7", "#22c55e"]
bars = ax.barh(languages, token_counts, color=colors)
ax.set_xlabel("Token Count")
ax.set_title("Tokens Required for Similar Content (cl100k_base)")
for bar, count in zip(bars, token_counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=10)
plt.tight_layout()
plt.show()

### 2.3 Counting Tokens Before API Calls (Save Money)

Always count tokens before sending a prompt to the API. This helps you:
1. Avoid exceeding context length limits.
2. Estimate costs before making expensive API calls.
3. Optimize prompt length for efficiency.

In [ ]:
def count_tokens(messages: list[dict], model: str = "gpt-4o") -> int:
    """Approximate token count for a list of chat messages."""
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # overhead per message (role, content markers)
        for value in msg.values():
            total += len(enc.encode(str(value)))
    total += 2  # reply priming tokens
    return total


messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain transformers in one paragraph."},
]
token_count = count_tokens(messages)
print(f"Estimated tokens: {token_count}")

# Cost estimation
input_price_per_million = 2.50   # GPT-4o input price (USD)
output_price_per_million = 10.00  # GPT-4o output price (USD)
estimated_output_tokens = 150

input_cost = token_count * input_price_per_million / 1_000_000
output_cost = estimated_output_tokens * output_price_per_million / 1_000_000
total_cost = input_cost + output_cost

print(f"\nCost estimate (GPT-4o):")
print(f"  Input:  {token_count} tokens x ${input_price_per_million}/1M = ${input_cost:.6f}")
print(f"  Output: ~{estimated_output_tokens} tokens x ${output_price_per_million}/1M = ${output_cost:.6f}")
print(f"  Total:  ${total_cost:.6f}")
print(f"\nAt scale (1M requests/day): ${total_cost * 1_000_000:.2f}/day")

### 2.4 Special Tokens

Special tokens signal structure to the model that cannot be expressed in plain text. They vary by model family:
- **GPT**: `<|endoftext|>`, `<|im_start|>`, `<|im_end|>`
- **Llama**: `<s>` (BOS), `</s>` (EOS)
- **BERT**: `[CLS]`, `[SEP]`, `[MASK]`

In [ ]:
# Special tokens must be explicitly allowed during encoding
chat_template = "<|im_start|>user\nHello, how are you?<|im_end|>"

# Without allowing special tokens, they are treated as literal text
normal_tokens = enc.encode(chat_template)
print(f"Without special tokens: {len(normal_tokens)} tokens")

# With special tokens enabled
special_tokens = enc.encode(chat_template, allowed_special="all")
print(f"With special tokens:    {len(special_tokens)} tokens")
print(f"\nSpecial token IDs: {special_tokens}")

# Decode to see the difference
print(f"\nDecoded (special): {enc.decode(special_tokens)}")

# Show vocabulary sizes across models
print("\nVocabulary sizes by model:")
vocab_info = [
    ("GPT-2 (r50k_base)",    50257),
    ("GPT-4 (cl100k_base)",  100277),
    ("Llama 3",              128256),
]
for name, size in vocab_info:
    print(f"  {name:<25} {size:>8,} tokens")

---
## Section 3: Embeddings

After tokenization, each integer ID is converted into a high-dimensional vector via an **embedding table** — a matrix of shape `(vocab_size, d_model)`. These vectors are the model's internal language.

Embedding spaces develop a **geometry of meaning**:
- Words with similar meanings cluster together ("cat" near "dog")
- Directions carry semantic meaning (`king - man + woman ≈ queen`)
- This is why **RAG** works: embed query + documents, find nearest neighbors

### 3.1 Generating Embeddings with OpenAI

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from os.environ


def get_embedding(text: str, model: str = "text-embedding-3-small") -> np.ndarray:
    """Call the OpenAI embedding API and return a unit-normalized numpy array."""
    text = text.replace("\n", " ")  # newlines degrade quality slightly
    response = client.embeddings.create(input=[text], model=model)
    vec = np.array(response.data[0].embedding)
    return vec / np.linalg.norm(vec)  # L2-normalize for cosine = dot product


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two unit-normalized vectors."""
    return float(np.dot(a, b))


# Test with a single embedding
test_vec = get_embedding("Hello, world!")
print(f"Embedding dimension: {test_vec.shape[0]}")
print(f"Embedding norm (should be ~1.0): {np.linalg.norm(test_vec):.4f}")
print(f"First 10 values: {test_vec[:10].round(4)}")

### 3.2 Semantic Similarity with Cosine Similarity

Cosine similarity measures the angle between two vectors:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \times \|\mathbf{b}\|}$$

Ranges from -1 (opposite) through 0 (unrelated) to 1 (identical).

In [ ]:
# Compare semantic similarity of sentence pairs
pairs = [
    ("The cat sat on the mat.",          "A feline rested on a rug."),
    ("Machine learning model deployment.", "Serving ML systems in production."),
    ("How do I make pasta carbonara?",    "Explain quantum entanglement."),
    ("king",                              "queen"),
    ("king",                              "automobile"),
]

print(f"{'Sentence A':<40}  {'Sentence B':<40}  {'Similarity':>10}")
print("-" * 95)
similarities = []
for a, b in pairs:
    emb_a = get_embedding(a)
    emb_b = get_embedding(b)
    sim = cosine_similarity(emb_a, emb_b)
    similarities.append(sim)
    print(f"{a[:38]:<40}  {b[:38]:<40}  {sim:>10.4f}")

print("\nHigh similarity = semantically related (even with different words).")
print("Low similarity = unrelated topics.")

### 3.3 Vector Arithmetic: king - man + woman = ?

A classic demonstration that embedding spaces encode semantic relationships as directions.

In [ ]:
# Embedding vector arithmetic
words = ["king", "man", "woman", "queen", "prince", "princess"]
embeddings = {w: get_embedding(w) for w in words}

# king - man + woman should be closest to queen
analogy = embeddings["king"] - embeddings["man"] + embeddings["woman"]
analogy = analogy / np.linalg.norm(analogy)  # Re-normalize after arithmetic

print("king - man + woman -> similarity to:")
results = []
for word, vec in embeddings.items():
    sim = cosine_similarity(analogy, vec)
    results.append((word, sim))
    print(f"  {word:10s}: {sim:.4f}")

# Sort by similarity
results.sort(key=lambda x: x[1], reverse=True)
print(f"\nClosest match: '{results[0][0]}' (expected: 'queen')")

### 3.4 Embedding Visualization (2D Projection)

We can project 1536-dimensional embeddings down to 2D using PCA to visualize semantic clusters.

In [ ]:
# Embed a variety of terms to see clustering
terms = [
    # Animals
    "cat", "dog", "elephant", "whale",
    # Technology
    "computer", "software", "algorithm", "database",
    # Food
    "pizza", "sushi", "pasta", "burger",
    # Emotions
    "happy", "sad", "angry", "excited",
]

categories = (
    ["Animals"] * 4 + ["Technology"] * 4 +
    ["Food"] * 4 + ["Emotions"] * 4
)

# Get embeddings for all terms
response = client.embeddings.create(input=terms, model="text-embedding-3-small")
all_embeddings = np.array([r.embedding for r in response.data])
print(f"Embedding matrix shape: {all_embeddings.shape}")

# PCA to 2D
centered = all_embeddings - all_embeddings.mean(axis=0)
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
coords_2d = centered @ Vt[:2].T  # Project onto first 2 principal components

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
category_colors = {"Animals": "#3b82f6", "Technology": "#22c55e",
                   "Food": "#f97316", "Emotions": "#a855f7"}

for i, (term, cat) in enumerate(zip(terms, categories)):
    ax.scatter(coords_2d[i, 0], coords_2d[i, 1],
              c=category_colors[cat], s=100, zorder=5)
    ax.annotate(term, (coords_2d[i, 0], coords_2d[i, 1]),
               textcoords="offset points", xytext=(8, 5),
               fontsize=11, color=category_colors[cat])

# Legend
for cat, color in category_colors.items():
    ax.scatter([], [], c=color, s=80, label=cat)
ax.legend(fontsize=11, loc="upper right")

ax.set_title("Embedding Space (PCA to 2D) — Semantic Clusters", fontsize=14)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Semantically related words cluster together in embedding space.")
print("This is why vector search (RAG) works — similar meanings = nearby vectors.")

### 3.5 Pairwise Similarity Matrix

A similarity matrix shows how every document relates to every other. This is the foundation of semantic search and retrieval.

In [ ]:
# Batch embed documents
documents = [
    "Transformers use self-attention to model sequences.",
    "Python is a dynamically-typed interpreted language.",
    "Gradient descent minimises the loss function iteratively.",
    "RAG retrieves relevant chunks before generating an answer.",
    "The attention mechanism computes weighted sums of values.",
    "JavaScript runs in web browsers and Node.js servers.",
]

response = client.embeddings.create(input=documents, model="text-embedding-3-small")
doc_embeddings = np.array([r.embedding for r in response.data])

# Normalize and compute pairwise cosine similarity
norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
unit_embs = doc_embeddings / norms
sim_matrix = unit_embs @ unit_embs.T

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap="RdYlBu_r", vmin=0, vmax=1)

short_labels = [d[:45] + "..." if len(d) > 45 else d for d in documents]
ax.set_xticks(range(len(documents)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(documents)))
ax.set_yticklabels(short_labels, fontsize=9)

# Annotate values
for i in range(len(documents)):
    for j in range(len(documents)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color="white" if sim_matrix[i, j] > 0.6 else "black")

plt.colorbar(im, ax=ax, shrink=0.8, label="Cosine Similarity")
ax.set_title("Pairwise Document Similarity (text-embedding-3-small)", fontsize=13)
plt.tight_layout()
plt.show()

print("Notice: attention/transformer docs are similar to each other,")
print("programming language docs cluster together, etc.")

---
## Section 4: Pre-training, Instruction Tuning & RLHF

Training a modern LLM happens in three stages:

| Stage | Objective | Data | Cost | Output |
|---|---|---|---|---|
| **Pre-training** | Predict next token (CLM) | Trillions of tokens from the web | $4M - $100M+ | Base/foundation model |
| **Instruction Tuning (SFT)** | Follow instructions | Thousands of (instruction, response) pairs | $1K - $100K | Helpful assistant |
| **RLHF / DPO** | Align with human preferences | Human preference comparisons | $10K - $1M | Safe, helpful, honest model |

### Why does this matter?
- **Base models** are powerful but "feral" — they complete text, not answer questions.
- **SFT** transforms them into assistants.
- **RLHF/DPO** shapes them to be helpful, harmless, and honest (Claude uses Constitutional AI).

### Scaling Laws
- Performance scales as a **power law** with compute, parameters, and data.
- **Chinchilla optimal**: ~20 tokens per parameter (8B model needs ~160B tokens).
- Llama 3 8B was trained on **15T tokens** — deliberately over-trained for better inference performance.

In [ ]:
# Visualize the training pipeline
fig, ax = plt.subplots(figsize=(14, 4))

stages = [
    {"name": "Pre-training\n(CLM)", "duration": 70, "color": "#3b82f6",
     "detail": "Trillions of tokens\nNext-token prediction\n$4M-$100M+"},
    {"name": "SFT\n(Instruction Tuning)", "duration": 15, "color": "#06b6d4",
     "detail": "~100K examples\nInstruction-response pairs\n$1K-$100K"},
    {"name": "RLHF / DPO\n(Alignment)", "duration": 10, "color": "#a855f7",
     "detail": "Human preferences\nReward model + PPO/DPO\n$10K-$1M"},
    {"name": "Deployed\nModel", "duration": 5, "color": "#22c55e",
     "detail": "ChatGPT, Claude,\nGemini, Llama Chat"},
]

x_pos = 0
for stage in stages:
    rect = plt.Rectangle((x_pos, 0), stage["duration"], 1,
                         facecolor=stage["color"], alpha=0.8,
                         edgecolor="white", linewidth=2)
    ax.add_patch(rect)
    ax.text(x_pos + stage["duration"] / 2, 0.7, stage["name"],
            ha="center", va="center", fontsize=10, fontweight="bold", color="white")
    ax.text(x_pos + stage["duration"] / 2, 0.28, stage["detail"],
            ha="center", va="center", fontsize=7.5, color="white", alpha=0.9)
    # Arrow between stages
    if x_pos > 0:
        ax.annotate("", xy=(x_pos, 0.5), xytext=(x_pos - 0.5, 0.5),
                    arrowprops=dict(arrowstyle="->", color="white", lw=2))
    x_pos += stage["duration"]

ax.set_xlim(-1, x_pos + 1)
ax.set_ylim(-0.1, 1.2)
ax.set_title("LLM Training Pipeline: From Random Weights to Production Model", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Chinchilla scaling law visualization
# Optimal tokens ~= 20 * parameters
param_counts = np.array([0.1, 0.5, 1, 3, 7, 8, 13, 30, 65, 70, 175])  # Billions
chinchilla_optimal = param_counts * 20  # Billions of tokens

# Actual training tokens for known models
actual_models = {
    "GPT-2 (1.5B)":      {"params": 1.5,   "tokens": 40},
    "GPT-3 (175B)":      {"params": 175,    "tokens": 300},
    "Chinchilla (70B)":  {"params": 70,     "tokens": 1400},
    "Llama 2 7B":        {"params": 7,      "tokens": 2000},
    "Llama 3 8B":        {"params": 8,      "tokens": 15000},
    "Llama 3 70B":       {"params": 70,     "tokens": 15000},
}

fig, ax = plt.subplots(figsize=(10, 6))

# Chinchilla optimal line
ax.plot(param_counts, chinchilla_optimal, "--", color="#ef4444", linewidth=2,
        label="Chinchilla Optimal (20x params)", alpha=0.7)

# Actual models
for name, info in actual_models.items():
    ax.scatter(info["params"], info["tokens"], s=100, zorder=5)
    ax.annotate(name, (info["params"], info["tokens"]),
               textcoords="offset points", xytext=(8, 5), fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Parameters (Billions)", fontsize=12)
ax.set_ylabel("Training Tokens (Billions)", fontsize=12)
ax.set_title("Scaling Laws: Parameters vs Training Tokens", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("Models above the red line are 'over-trained' relative to Chinchilla optimal.")
print("Llama 3 deliberately over-trains to create a smaller model that performs well at inference.")

---
## Section 5: Inference Mechanics — Temperature & Sampling

During inference, the model generates text **autoregressively** — one token at a time. Each step produces a probability distribution over the entire vocabulary (~100K tokens). How you sample from this distribution determines the output quality.

Key parameters:
- **Temperature** — controls randomness. T=0 is deterministic (greedy), T=1 is natural, T>1 is chaotic.
- **Top-k** — only consider the k most likely tokens.
- **Top-p (nucleus)** — keep the smallest set of tokens whose cumulative probability exceeds p.
- **Greedy decoding** — always pick the most likely token (temperature=0).

In [ ]:
def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Apply temperature scaling then softmax."""
    scaled = logits / max(temperature, 1e-8)
    exp = np.exp(scaled - scaled.max())  # numerical stability
    return exp / exp.sum()


def top_k_filter(logits: np.ndarray, k: int) -> np.ndarray:
    """Zero out all logits except the top-k."""
    threshold = np.sort(logits)[-k]
    filtered = np.where(logits >= threshold, logits, -np.inf)
    return filtered


def top_p_filter(logits: np.ndarray, p: float) -> np.ndarray:
    """Keep the smallest set of tokens whose cumulative probability >= p."""
    probs = softmax(logits)
    sorted_idx = np.argsort(probs)[::-1]  # descending
    cumulative = np.cumsum(probs[sorted_idx])
    cutoff = np.searchsorted(cumulative, p) + 1
    keep = sorted_idx[:cutoff]
    filtered = np.full_like(logits, -np.inf, dtype=float)
    filtered[keep] = logits[keep]
    return filtered


def sample_token(logits: np.ndarray, temperature: float = 1.0,
                 top_k: int = 0, top_p: float = 1.0) -> int:
    """
    Full sampling pipeline:
    1. Apply top-k filter (if k > 0)
    2. Apply top-p (nucleus) filter
    3. Apply temperature and softmax
    4. Sample from resulting distribution
    """
    if top_k > 0:
        logits = top_k_filter(logits, top_k)
    if top_p < 1.0:
        logits = top_p_filter(logits, top_p)
    probs = softmax(logits, temperature)
    return int(np.random.choice(len(probs), p=probs))


print("Sampling functions defined: softmax, top_k_filter, top_p_filter, sample_token")

### 5.1 Effect of Temperature on Token Probabilities

Temperature divides logits before softmax: `p_i = softmax(logits / T)`
- **T < 1** sharpens the distribution (more deterministic, "confident")
- **T = 1** is the natural learned distribution
- **T > 1** flattens the distribution (more random, creative but risky)

In [ ]:
# Simulate raw logits for a small vocabulary
np.random.seed(0)
vocab_labels = ["the", "a", "cat", "dog", "sat", "ran", "on", "big", "red", "mat"]
vocab_size = len(vocab_labels)
logits = np.array([3.5, 1.2, 2.8, 0.5, 2.1, -0.3, 1.8, 0.1, -1.0, 1.5])

temperatures = [0.2, 0.5, 0.7, 1.0, 1.5, 2.0]

# Print probability table
print(f"{'Token':<8}", end="")
for T in temperatures:
    print(f"  T={T:<4}", end="")
print()
print("-" * 65)

prob_data = {}
for T in temperatures:
    prob_data[T] = softmax(logits, T)

for i in range(vocab_size):
    print(f"  {vocab_labels[i]:<6}", end="")
    for T in temperatures:
        p = prob_data[T][i]
        print(f"  {p:.3f} ", end="")
    print()

print("\nAt T=0.2, 'the' dominates (almost greedy). At T=2.0, distribution is nearly uniform.")

In [ ]:
# Visualize temperature effect as bar charts
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, (T, ax) in enumerate(zip(temperatures, axes.flat)):
    probs = prob_data[T]
    sorted_idx = np.argsort(probs)[::-1]
    sorted_labels = [vocab_labels[i] for i in sorted_idx]
    sorted_probs = probs[sorted_idx]

    colors = plt.cm.Blues(np.linspace(0.9, 0.3, vocab_size))
    ax.barh(range(vocab_size), sorted_probs, color=colors)
    ax.set_yticks(range(vocab_size))
    ax.set_yticklabels(sorted_labels, fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_title(f"Temperature = {T}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Probability")
    ax.invert_yaxis()

    # Entropy (measure of randomness)
    entropy = -np.sum(probs * np.log2(probs + 1e-10))
    ax.text(0.95, 0.95, f"Entropy: {entropy:.2f} bits",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

plt.suptitle("Effect of Temperature on Token Probability Distribution",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Higher entropy = more randomness. Use T=0-0.3 for factual tasks, T=0.7-1.0 for creative tasks.")

### 5.2 Comparing Sampling Strategies

Different sampling strategies affect which tokens can be selected:
- **Greedy**: Always pick the top token (deterministic).
- **Top-k**: Only consider the k most likely tokens.
- **Top-p (nucleus)**: Dynamically size the candidate set by cumulative probability.
- **Temperature + Top-p**: The most common production configuration.

In [ ]:
# Show how each strategy modifies the probability distribution
np.random.seed(42)

strategies = [
    {"name": "Original (T=1.0)",   "fn": lambda l: softmax(l, 1.0)},
    {"name": "Greedy (T=0.01)",    "fn": lambda l: softmax(l, 0.01)},
    {"name": "Top-k (k=3)",        "fn": lambda l: softmax(top_k_filter(l, 3))},
    {"name": "Top-p (p=0.8)",      "fn": lambda l: softmax(top_p_filter(l, 0.8))},
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for strat, ax in zip(strategies, axes):
    probs = strat["fn"](logits)
    colors = ["#3b82f6" if p > 0.001 else "#e5e7eb" for p in probs]
    ax.bar(vocab_labels, probs, color=colors)
    ax.set_title(strat["name"], fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Probability")
    ax.tick_params(axis="x", rotation=45)

    # Count eligible tokens
    eligible = sum(1 for p in probs if p > 0.001)
    ax.text(0.95, 0.95, f"{eligible} eligible",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

plt.suptitle("Sampling Strategies: How They Filter Token Probabilities",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate sampling variability across runs
n_samples = 1000
configs = [
    {"name": "Greedy (T=0.01)",       "temperature": 0.01, "top_k": 0, "top_p": 1.0},
    {"name": "Low temp (T=0.3)",       "temperature": 0.3,  "top_k": 0, "top_p": 1.0},
    {"name": "Medium (T=0.7, p=0.9)",  "temperature": 0.7,  "top_k": 0, "top_p": 0.9},
    {"name": "High temp (T=1.5)",      "temperature": 1.5,  "top_k": 0, "top_p": 1.0},
]

print(f"Sampling {n_samples} tokens per strategy:\n")
print(f"{'Strategy':<28} {'Top token %':>12} {'Unique tokens':>14} {'Entropy':>8}")
print("-" * 65)

for config in configs:
    samples = []
    for _ in range(n_samples):
        tok = sample_token(logits, config["temperature"], config["top_k"], config["top_p"])
        samples.append(tok)

    counts = np.bincount(samples, minlength=vocab_size)
    top_pct = counts.max() / n_samples * 100
    unique = np.sum(counts > 0)

    # Empirical entropy
    emp_probs = counts / n_samples
    emp_probs = emp_probs[emp_probs > 0]
    entropy = -np.sum(emp_probs * np.log2(emp_probs))

    print(f"{config['name']:<28} {top_pct:>11.1f}% {unique:>14} {entropy:>8.2f}")

print("\nRecommendations:")
print("  Factual Q&A / Code:  T=0.0 to 0.3 (deterministic, accurate)")
print("  General chat:        T=0.7, top_p=0.9 (balanced)")
print("  Creative writing:    T=0.9 to 1.0 (varied, surprising)")

### 5.3 KV Cache Memory Estimation

The **KV cache** stores Key and Value matrices for all previously generated tokens, avoiding recomputation. Its size is a critical constraint on how many concurrent users a GPU can serve.

Size = `2 (K+V) x layers x heads x d_head x seq_len x batch x dtype_bytes`

In [ ]:
def kv_cache_bytes(n_layers: int, n_kv_heads: int, d_head: int,
                   seq_len: int, batch_size: int = 1,
                   dtype_bytes: int = 2) -> int:
    """
    Estimate KV cache memory in bytes.

    KV cache stores K and V for each layer, KV head, and token.
    Shape: 2 (K+V) x layers x kv_heads x seq_len x d_head x batch x dtype
    """
    return 2 * n_layers * n_kv_heads * d_head * seq_len * batch_size * dtype_bytes


# Compare KV cache sizes for different models and context lengths
model_configs = [
    {"name": "Llama 3 8B",        "layers": 32, "kv_heads": 8,  "d_head": 128},
    {"name": "Llama 3 70B",       "layers": 80, "kv_heads": 8,  "d_head": 128},
    {"name": "GPT-4 class (est.)", "layers": 96, "kv_heads": 96, "d_head": 128},
]

context_lengths = [2048, 8192, 32768, 131072]

print(f"{'Model':<22}", end="")
for ctx in context_lengths:
    print(f"  {ctx//1024:>4}K ctx", end="")
print()
print("-" * 62)

for model in model_configs:
    print(f"{model['name']:<22}", end="")
    for ctx in context_lengths:
        size = kv_cache_bytes(model["layers"], model["kv_heads"],
                             model["d_head"], ctx, batch_size=1, dtype_bytes=2)
        if size >= 1e9:
            print(f"  {size/1e9:>6.1f} GB", end="")
        else:
            print(f"  {size/1e6:>6.0f} MB", end="")
    print()

print("\nNote: Llama 3 uses GQA (Grouped-Query Attention) with only 8 KV heads,")
print("which dramatically reduces KV cache compared to full multi-head attention.")
print("\nA100 80GB has ~80GB HBM. After model weights, remaining memory constrains")
print("how many concurrent long-context requests can be served simultaneously.")

### 5.4 Context Window and the Lost-in-the-Middle Problem

Even with 128K context windows, models recall information much better from the **beginning** and **end** of the context than from the **middle**. This is why structured retrieval (RAG) often outperforms naive context stuffing.

In [ ]:
# Simulated "lost in the middle" recall curve
# Based on findings from Liu et al., 2023 "Lost in the Middle"
positions = np.linspace(0, 1, 100)  # Normalized position in context (0=start, 1=end)

# U-shaped recall curve: high at start, low in middle, high at end
recall = 0.4 + 0.5 * (np.exp(-10 * positions) + np.exp(-10 * (1 - positions)))
recall = np.clip(recall, 0, 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(positions * 100, recall * 100, alpha=0.3, color="#3b82f6")
ax.plot(positions * 100, recall * 100, color="#3b82f6", linewidth=2.5)

ax.axhline(y=90, color="#22c55e", linestyle="--", alpha=0.5, label="RAG retrieval (relevant chunks only)")

ax.set_xlabel("Position in Context Window (%)", fontsize=12)
ax.set_ylabel("Information Recall Accuracy (%)", fontsize=12)
ax.set_title('"Lost in the Middle" — Why RAG Beats Context Stuffing', fontsize=13)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)

ax.annotate("High recall\n(beginning)", xy=(5, 90), fontsize=10, color="#3b82f6")
ax.annotate("Low recall\n(middle)", xy=(45, 45), fontsize=10, color="#ef4444")
ax.annotate("High recall\n(end)", xy=(85, 85), fontsize=10, color="#3b82f6")

ax.legend(fontsize=10, loc="lower center")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: Even with a 128K context window, information at position 64K")
print("may be effectively ignored. RAG retrieves only relevant chunks, keeping them")
print("at the top of the context where recall is highest.")

---
## Summary

In this notebook we covered the five pillars of modern GenAI:

1. **Transformer Architecture** — Scaled dot-product attention, multi-head attention, residual connections, feed-forward networks, and the full transformer block implemented from scratch in NumPy.

2. **Tokenization** — BPE with tiktoken, token counting, multilingual efficiency, special tokens, and cost estimation.

3. **Embeddings** — OpenAI embedding API, cosine similarity, vector arithmetic (`king - man + woman`), 2D visualization with PCA, and pairwise similarity matrices.

4. **Training Pipeline** — Pre-training (CLM), instruction tuning (SFT), RLHF/DPO alignment, and Chinchilla scaling laws.

5. **Inference Mechanics** — Temperature, top-k, top-p sampling strategies, KV cache estimation, and the lost-in-the-middle problem.

**Next notebook**: Module 02 covers LLMs, SLMs, and Multimodal Models — the different model families you will use throughout the course.